# 14 — Expert (Inter-Observer) Variability

This notebook demonstrates the framework for comparing manual
annotations from multiple expert reviewers.

Metrics:
- Mean difference
- SD of differences
- Agreement rates at ±5, ±10, ±20 ms


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecg_analytics.validation.expert_variability import (
    pairwise_comparison,
    expert_variability_report,
)


## Simulate expert annotations


In [ ]:
np.random.seed(42)
n_beats = 50
# Ground-truth T-end positions (ms)
truth = np.random.normal(420, 15, n_beats)

# Three reviewers with different biases and noise
reviewer_A = truth + np.random.normal(0, 3, n_beats)
reviewer_B = truth + np.random.normal(2, 5, n_beats)
reviewer_C = truth + np.random.normal(-1, 4, n_beats)

annotations = {
    'Reviewer A': reviewer_A,
    'Reviewer B': reviewer_B,
    'Reviewer C': reviewer_C,
}


## Pairwise comparison


In [ ]:
comp = pairwise_comparison(reviewer_A, reviewer_B, 'A', 'B')
print(f'A vs B: mean diff = {comp.mean_diff_ms:.2f} ms, '
      f'SD = {comp.sd_diff_ms:.2f} ms')
print(f'  Agreement ±5 ms: {comp.agreement_within_5ms:.1%}')
print(f'  Agreement ±10 ms: {comp.agreement_within_10ms:.1%}')


## Full variability report


In [ ]:
report = expert_variability_report(annotations)
print(f'Reviewers: {report.n_reviewers}')
print(f'Beats: {report.n_beats}')
print(f'Overall SD: {report.overall_sd_ms:.2f} ms')
print()
for comp in report.pairwise:
    print(f'{comp.reviewer_a} vs {comp.reviewer_b}: '
          f'mean={comp.mean_diff_ms:.2f}, SD={comp.sd_diff_ms:.2f}, '
          f'±5ms={comp.agreement_within_5ms:.1%}')


## Visualization


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [(reviewer_A, reviewer_B, 'A vs B'),
         (reviewer_A, reviewer_C, 'A vs C'),
         (reviewer_B, reviewer_C, 'B vs C')]
for ax, (a, b, title) in zip(axes, pairs):
    diff = a - b
    mean = (a + b) / 2
    ax.scatter(mean, diff, s=12, alpha=0.6)
    ax.axhline(np.mean(diff), color='red', linestyle='--')
    ax.axhline(np.mean(diff) + 1.96 * np.std(diff), color='gray', linestyle=':')
    ax.axhline(np.mean(diff) - 1.96 * np.std(diff), color='gray', linestyle=':')
    ax.set_title(title)
    ax.set_xlabel('Mean (ms)')
    ax.set_ylabel('Difference (ms)')
plt.tight_layout()
plt.show()
